In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib

In [3]:
CICFLOWMETER_TO_CIC_COLUMNS = {
    'src_ip': 'Source IP',
    'dst_ip': 'Destination IP',
    'src_port': 'Source Port',
    'dst_port': 'Destination Port',
    'protocol': 'Protocol',
    'timestamp': 'Timestamp',
    'flow_duration': 'Flow Duration',
    'flow_byts_s': 'Flow Bytes/s',
    'flow_pkts_s': 'Flow Packets/s',
    'fwd_pkts_s': 'Fwd Packets/s',
    'bwd_pkts_s': 'Bwd Packets/s',
    'tot_fwd_pkts': 'Total Fwd Packets',
    'tot_bwd_pkts': 'Total Backward Packets',
    'totlen_fwd_pkts': 'Total Length of Fwd Packets',
    'totlen_bwd_pkts': 'Total Length of Bwd Packets',
    'fwd_pkt_len_max': 'Fwd Packet Length Max',
    'fwd_pkt_len_min': 'Fwd Packet Length Min',
    'fwd_pkt_len_mean': 'Fwd Packet Length Mean',
    'fwd_pkt_len_std': 'Fwd Packet Length Std',
    'bwd_pkt_len_max': 'Bwd Packet Length Max',
    'bwd_pkt_len_min': 'Bwd Packet Length Min',
    'bwd_pkt_len_mean': 'Bwd Packet Length Mean',
    'bwd_pkt_len_std': 'Bwd Packet Length Std',
    'pkt_len_max': 'Max Packet Length',
    'pkt_len_min': 'Min Packet Length',
    'pkt_len_mean': 'Packet Length Mean',
    'pkt_len_std': 'Packet Length Std',
    'pkt_len_var': 'Packet Length Variance',
    'fwd_header_len': 'Fwd Header Length',
    'bwd_header_len': 'Bwd Header Length',
    'fwd_seg_size_min': 'min_seg_size_forward',
    'fwd_act_data_pkts': 'act_data_pkt_fwd',
    'flow_iat_mean': 'Flow IAT Mean',
    'flow_iat_max': 'Flow IAT Max',
    'flow_iat_min': 'Flow IAT Min',
    'flow_iat_std': 'Flow IAT Std',
    'fwd_iat_tot': 'Fwd IAT Total',
    'fwd_iat_max': 'Fwd IAT Max',
    'fwd_iat_min': 'Fwd IAT Min',
    'fwd_iat_mean': 'Fwd IAT Mean',
    'fwd_iat_std': 'Fwd IAT Std',
    'bwd_iat_tot': 'Bwd IAT Total',
    'bwd_iat_max': 'Bwd IAT Max',
    'bwd_iat_min': 'Bwd IAT Min',
    'bwd_iat_mean': 'Bwd IAT Mean',
    'bwd_iat_std': 'Bwd IAT Std',
    'fwd_psh_flags': 'Fwd PSH Flags',
    'bwd_psh_flags': 'Bwd PSH Flags',
    'fwd_urg_flags': 'Fwd URG Flags',
    'bwd_urg_flags': 'Bwd URG Flags',
    'fin_flag_cnt': 'FIN Flag Count',
    'syn_flag_cnt': 'SYN Flag Count',
    'rst_flag_cnt': 'RST Flag Count',
    'psh_flag_cnt': 'PSH Flag Count',
    'ack_flag_cnt': 'ACK Flag Count',
    'urg_flag_cnt': 'URG Flag Count',
    'ece_flag_cnt': 'ECE Flag Count',
    'cwr_flag_count': 'CWE Flag Count',   
    'down_up_ratio': 'Down/Up Ratio',
    'pkt_size_avg': 'Average Packet Size',
    'fwd_seg_size_avg': 'Avg Fwd Segment Size',
    'bwd_seg_size_avg': 'Avg Bwd Segment Size',
    'fwd_byts_b_avg': 'Fwd Avg Bytes/Bulk',
    'fwd_pkts_b_avg': 'Fwd Avg Packets/Bulk',
    'fwd_blk_rate_avg': 'Fwd Avg Bulk Rate',
    'bwd_byts_b_avg': 'Bwd Avg Bytes/Bulk',
    'bwd_pkts_b_avg': 'Bwd Avg Packets/Bulk',
    'bwd_blk_rate_avg': 'Bwd Avg Bulk Rate',
    'subflow_fwd_pkts': 'Subflow Fwd Packets',
    'subflow_bwd_pkts': 'Subflow Bwd Packets',
    'subflow_fwd_byts': 'Subflow Fwd Bytes',
    'subflow_bwd_byts': 'Subflow Bwd Bytes',
    'init_fwd_win_byts': 'Init_Win_bytes_forward',
    'init_bwd_win_byts': 'Init_Win_bytes_backward',
    'active_max': 'Active Max',
    'active_min': 'Active Min',
    'active_mean': 'Active Mean',
    'active_std': 'Active Std',
    'idle_max': 'Idle Max',
    'idle_min': 'Idle Min',
    'idle_mean': 'Idle Mean',
    'idle_std': 'Idle Std',
}

In [4]:
# Time-domain columns that cicflowmeter reports in SECONDS but CIC-IDS2017 (what the model was trained on) reports in MICROSECONDS
TIME_COLUMNS_TO_FIX = [
    'Flow Duration',
    'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Flow IAT Std',
    'Fwd IAT Total', 'Fwd IAT Max', 'Fwd IAT Min', 'Fwd IAT Mean', 'Fwd IAT Std',
    'Bwd IAT Total', 'Bwd IAT Max', 'Bwd IAT Min', 'Bwd IAT Mean', 'Bwd IAT Std',
    'Active Max', 'Active Min', 'Active Mean', 'Active Std',
    'Idle Max', 'Idle Min', 'Idle Mean', 'Idle Std',
]
# NOT included: Flow Bytes/s, Flow Packets/s, Fwd Packets/s, Bwd Packets/s — these are true per-second rates, already correctly scaled

In [5]:
IDENTIFIER_COLS = ['Source IP', 'Destination IP', 'Source Port', 'Protocol', 'Timestamp']
# These never existed in the training data at all, so they're dropped entirely, not just excluded like Destination Port.

In [6]:
def clean_and_prepare(path, drop_near_zero_duration=True):
    df = pd.read_csv(path)
    df = df.rename(columns=CICFLOWMETER_TO_CIC_COLUMNS)

    # Fix: drop the near-zero-duration bug flows, BEFORE unit conversion, while Duration is still in seconds.
    if drop_near_zero_duration:
        df = df[df['Flow Duration'] >= 0.0001].reset_index(drop=True)

    # Fix the seconds -> microseconds mismatch
    for col in TIME_COLUMNS_TO_FIX:
        df[col] = df[col] * 1_000_000

    return df

In [7]:
df1 = clean_and_prepare(r"C:\glassIDS\data\processed\self_captured\attack1_flows.csv", drop_near_zero_duration=False)
df2 = clean_and_prepare(r"C:\glassIDS\data\processed\self_captured\attack2_flows.csv")
df3 = clean_and_prepare(r"C:\glassIDS\data\processed\self_captured\attack3_flows.csv")

In [8]:
#sanity check for df1 as it has lots of small flow durations
print(df1['Flow Duration'].describe())

print(df1['Flow Duration'].nlargest(5))

inf_check = df1[['Flow Bytes/s', 'Flow Packets/s']].replace([np.inf, -np.inf], np.nan).isna().sum()
print(inf_check)

count    6.553600e+04
mean     2.543181e+03
std      6.395321e+05
min      2.000000e+00
25%      1.900000e+01
50%      3.700000e+01
75%      6.200000e+01
max      1.637203e+08
Name: Flow Duration, dtype: float64
0        163720263.0
28232         6557.0
28231         6544.0
28230         6531.0
28229         6516.0
Name: Flow Duration, dtype: float64
Flow Bytes/s      0
Flow Packets/s    0
dtype: int64


In [9]:
def label_attack1(df):
    df = df.copy()
    df['category'] = 'attack_portscan'
    is_control = (df['Destination Port'] == 22) & (df['Flow Duration'] > 60_000_000)
    df.loc[is_control, 'category'] = 'benign_control'
    return df

def label_attack2(df):
    df = df.copy()
    df['category'] = 'attack_bruteforce'
    earliest_idx = df['Timestamp'].idxmin()  # the one flow present from t=0
    df.loc[earliest_idx, 'category'] = 'benign_control'
    return df

def label_attack3(df):
    df = df.copy()
    df['category'] = 'unlabeled'
    port22 = (df['Destination Port'] == 22) | (df['Source Port'] == 22)
    port8000 = (df['Destination Port'] == 8000) | (df['Source Port'] == 8000)
    df.loc[port22 & (df['Flow Duration'] < 10_000_000), 'category'] = 'benign_control_typed'
    df.loc[port22 & (df['Flow Duration'] >= 10_000_000), 'category'] = 'benign_control_logstream'
    df.loc[port8000 & (df['Flow Duration'] > 60_000_000), 'category'] = 'attack_slowdos'
    df.loc[port8000 & (df['Flow Duration'] <= 60_000_000), 'category'] = 'attack_probe'
    return df

df1 = label_attack1(df1)
df2 = label_attack2(df2)
df3 = label_attack3(df3)

In [10]:
# Sanity check before trusting the labels — every row should have been caught
assert (df3['category'] != 'unlabeled').all(), "Some Attack 3 flows didn't match any category — investigate before scoring"

combined = pd.concat([df1, df2, df3], ignore_index=True)
print(combined['category'].value_counts())


category
attack_portscan             65535
attack_slowdos               1000
attack_probe                   41
attack_bruteforce              14
benign_control                  2
benign_control_typed            1
benign_control_logstream        1
Name: count, dtype: int64


In [11]:
# --- Load the saved model and score ---
MODEL_DIR = r"C:\glassIDS\src\models"
model = xgb.XGBClassifier()
model.load_model(rf"{MODEL_DIR}\xgboost_baseline.json")
le = joblib.load(rf"{MODEL_DIR}\label_encoder.joblib")

drop_cols = IDENTIFIER_COLS + ['Destination Port', 'category']
feature_cols = [c for c in combined.columns if c not in drop_cols]

X = combined[feature_cols]

In [12]:
model_features = model.get_booster().feature_names
X_features = list(X.columns)

print(f"Model expects {len(model_features)} features, X has {len(X_features)}")
print("In model but missing from X:", set(model_features) - set(X_features))
print("In X but not expected by model:", set(X_features) - set(model_features))

Model expects 77 features, X has 76
In model but missing from X: {'Fwd Header Length.1'}
In X but not expected by model: set()


In [13]:
X['Fwd Header Length.1'] = X['Fwd Header Length']
X = X[model_features]  # reorder to match afterward either way

In [14]:
assert list(X.columns) == model.get_booster().feature_names, \
    "Column order doesn't match training — model will misread features silently"

pred_idx = model.predict(X)
combined['predicted_class'] = le.inverse_transform(pred_idx)

# --- Results: detection rate per attack category, false positives per benign category ---
results = combined.groupby(['category', 'predicted_class']).size().unstack(fill_value=0)
print(results)

predicted_class           BENIGN
category                        
attack_bruteforce             14
attack_portscan            65535
attack_probe                  41
attack_slowdos              1000
benign_control                 2
benign_control_logstream       1
benign_control_typed           1


In [15]:
sample = X[combined['category'] == 'attack_portscan'].iloc[:5]
print(model.predict_proba(sample))

[[9.99980450e-01 1.79215203e-08 1.19306875e-07 7.00844316e-09
  4.86384018e-08 1.44127119e-08 4.85077720e-08 1.21789526e-07
  1.31803422e-08 5.66281031e-08 1.72391265e-05 6.41260840e-08
  9.35065771e-07 1.33620857e-08 8.92677519e-07]
 [9.99990702e-01 8.57014282e-09 6.28733474e-08 3.89880261e-09
  2.13720028e-08 6.78811940e-09 2.30626735e-08 6.16418561e-08
  7.33226324e-09 3.00640259e-08 8.06224216e-06 3.93458848e-08
  5.20178901e-07 6.16033935e-09 4.96598204e-07]
 [9.99965787e-01 1.05206253e-08 7.71827118e-08 4.78613282e-09
  2.62360658e-08 8.41827408e-09 3.10715329e-08 7.56709539e-08
  9.00101593e-09 3.69063109e-08 3.26749396e-05 4.94741599e-08
  5.62340176e-07 1.62718052e-08 6.09619178e-07]
 [9.99964476e-01 1.56309383e-08 1.01168446e-07 6.27351060e-09
  3.43893980e-08 1.31694513e-08 3.30045964e-08 9.91868845e-08
  1.17981998e-08 4.83756146e-08 3.34881588e-05 7.32743359e-08
  7.37096116e-07 1.94630765e-08 7.99067891e-07]
 [9.99972939e-01 1.29927926e-08 8.40933509e-08 5.21467447e-09
  

In [26]:
cic = pd.read_csv(r"C:\glassIDS\data\processed\cicids2017_cleaned.csv")

compare_features = ['Flow Duration', 'Flow Bytes/s', 'Fwd Packets/s', 'Bwd Packets/s', 'Init_Win_bytes_forward']

# --- Attack 1 vs PortScan ---
cic_portscan = cic[cic['Label'] == 'PortScan']
self_portscan = df1[df1['category'] == 'attack_portscan']

print("=== CIC-IDS2017's own PortScan training rows ===")
print(cic_portscan[compare_features].describe())
print(f"\n=== Your self-captured PortScan flows (n={len(self_portscan)}) ===")
print(self_captured_scan[compare_features].describe())

# --- Attack 2 vs SSH-Patator ---
cic_sshpatator = cic[cic['Label'] == 'SSH-Patator']
self_bruteforce = df2[df2['category'] == 'attack_bruteforce']

print("=== CIC-IDS2017 SSH-Patator (training) ===")
print(cic_sshpatator[compare_features].describe())
print(f"\n=== Your self-captured brute-force (n={len(self_bruteforce)}) ===")
print(self_bruteforce[compare_features].describe())

# --- Attack 3 vs both slow-DoS classes (slowhttptest -H mimics both patterns) ---
cic_slowhttptest = cic[cic['Label'] == 'DoS Slowhttptest']
cic_slowloris = cic[cic['Label'] == 'DoS slowloris']
self_slowdos = df3[df3['category'] == 'attack_slowdos']

print("\n=== CIC-IDS2017 DoS Slowhttptest (training) ===")
print(cic_slowhttptest[compare_features].describe())
print("\n=== CIC-IDS2017 DoS slowloris (training) ===")
print(cic_slowloris[compare_features].describe())
print(f"\n=== Your self-captured slow-DoS (n={len(self_slowdos)}) ===")
print(self_slowdos[compare_features].describe())

=== CIC-IDS2017's own PortScan training rows ===
       Flow Duration  Flow Bytes/s   Fwd Packets/s   Bwd Packets/s  \
count   9.069400e+04  9.069400e+04    90694.000000    90694.000000   
mean    1.450965e+05  2.537042e+05    36009.744908    35985.646433   
std     3.077444e+06  5.674908e+05    79027.278989    79037.431274   
min     1.000000e+00  0.000000e+00        0.016693        0.000000   
25%     3.300000e+01  1.012658e+05    14705.882350    14705.882350   
50%     5.000000e+01  1.379310e+05    20000.000000    20000.000000   
75%     6.800000e+01  2.105263e+05    30303.030300    30303.030300   
max     1.198097e+08  8.000000e+06  1000000.000000  1000000.000000   

       Init_Win_bytes_forward  
count            90694.000000  
mean             15206.790714  
std              14116.949222  
min                 -1.000000  
25%               1024.000000  
50%              29200.000000  
75%              29200.000000  
max              64240.000000  

=== Your self-captured PortScan